# 🌞 DT-HRES-S: Digital Twin Prototype
## Hybrid Renewable Energy System for Mexican Indigenous Communities

**EPICS in IEEE 2025-2026** — *Student-Led Open-Source Digital Twin*

This notebook demonstrates the **end-to-end pipeline**:
1. Load TMY (Typical Meteorological Year) data for 4 Mexican cities
2. Simulate a PV + wind + battery system using **physics-based models**
3. Train and compare **4 ML algorithms** (Decision Tree, Random Forest, SVM, Neural Network)
4. Validate the digital twin with cross-city testing

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

## 0 — Setup (Colab users run this; local users skip)

In [5]:
!git clone https://github.com/Aaron-Cuevas/DT-HRES-S/tree/main/DT-HRES-S-v0.2.0
%cd DT-HRES-S
!pip install -q -r requirements.txt

fatal: destination path 'DT-HRES-S-v0.2.0' already exists and is not an empty directory.
[Errno 2] No such file or directory: 'DT-HRES-S'
/content/DT-HRES-S/DT-HRES-S
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [4]:

import sys
import os
# When running locally from notebooks/, add the repo root to the path
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_city, load_all_cities, get_summary, CITIES
from src import pv_model, wind_model, battery_model
from src import hres_simulator as hres
from src import ml_models

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

ModuleNotFoundError: No module named 'src'

## 1 — Explore the TMY dataset

Four Mexican cities spanning different climates: semi-arid (Monterrey), tropical (Campeche), highland (Mexico City), desert (San Ignacio).

In [1]:
summary = get_summary()
summary

NameError: name 'get_summary' is not defined

In [ ]:
# Plot monthly GHI for all cities
df_all = load_all_cities()
monthly = df_all.groupby(['city', 'month'])['ghi_Wm2'].mean().unstack(level=0)

fig, ax = plt.subplots(figsize=(11, 5))
monthly.plot(ax=ax, marker='o', linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Mean GHI (W/m²)')
ax.set_title('Solar resource seasonality — 4 Mexican cities')
ax.legend(title='City')
plt.tight_layout()
plt.show()

## 2 — Configure the HRES

We size a small community system suitable for Ixil-scale loads.

In [ ]:
config = hres.HRESConfig(
    pv=pv_model.PVSystem(
        p_rated_W=400,    # 400 W panel
        n_panels=10,      # 4 kWp array
        tilt_deg=20,      # rule of thumb: tilt ≈ latitude
        azimuth_deg=180,  # south-facing
    ),
    wind=wind_model.WindTurbine(
        rated_power_W=3000,    # 3 kW small turbine
        rotor_diameter_m=4.0,
        hub_height_m=12.0,
    ),
    battery=battery_model.Battery(
        capacity_kWh=10,       # 10 kWh Li-ion bank
        p_max_charge_kW=3.0,
        p_max_discharge_kW=3.0,
    ),
)
print(f'System: {config.pv.p_array_W/1000:.1f} kWp PV + {config.wind.rated_power_W/1000:.0f} kW Wind + {config.battery.capacity_kWh:.0f} kWh battery')

## 3 — Run the physics-based simulation

Generates the **ground-truth labels** that the ML twin will learn.

In [ ]:
df_mty = load_city('Monterrey')
sim = hres.run(df_mty, config)
kpis = hres.summarize(sim, config)

print('=== Annual KPIs (Monterrey) ===')
for k, v in kpis.items():
    fmt = f'{v:.2f}' if isinstance(v, float) else str(v)
    print(f'  {k:25s}: {fmt}')

In [ ]:
# Plot a typical week
week = sim.iloc[24*150:24*157]  # late May

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
axes[0].plot(week.index, week['p_pv_W']/1000, label='PV', color='gold')
axes[0].plot(week.index, week['p_wind_W']/1000, label='Wind', color='steelblue')
axes[0].plot(week.index, week['p_demand_W']/1000, label='Demand', color='black', alpha=0.7)
axes[0].set_ylabel('Power (kW)')
axes[0].legend()
axes[0].set_title('Typical week — supply, demand & dispatch')

axes[1].fill_between(week.index, 0, week['soc']*100, alpha=0.5, color='green')
axes[1].set_ylabel('Battery SoC (%)')
axes[1].set_ylim(0, 100)

axes[2].plot(week.index, week['p_unserved_W']/1000, color='red', label='Unmet load')
axes[2].plot(week.index, week['p_curtailed_W']/1000, color='orange', label='Curtailed')
axes[2].set_ylabel('kW')
axes[2].set_xlabel('Hour of year')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4 — Train the four candidate ML models (Task 3.3)

**Target:** hourly PV output `p_pv_W`. The same approach can predict total system power, battery SoC, or unmet load.

In [ ]:
# Single-city benchmark
results_mty = ml_models.benchmark(sim, target_col='p_pv_W')
print('Monterrey — random 80/20 split:')
results_mty.round(3)

In [ ]:
# Build the multi-city training set: simulate all 4 cities
all_sims = []
for city in CITIES.keys():
    df_city = load_city(city)
    sim_city = hres.run(df_city, config)
    all_sims.append(sim_city)
df_all_sim = pd.concat(all_sims, ignore_index=True)
print(f'Combined training set: {len(df_all_sim):,} samples across {df_all_sim["city"].nunique()} cities')

## 5 — Cross-city validation (Task 3.4)

**Leave-one-city-out:** train on 3 cities, test on the 4th. This simulates the realistic case of deploying the twin to a new community (like **Ixil, Yucatán**) where the model has never seen local data.

In [ ]:
cv_results = ml_models.leave_one_city_out(df_all_sim, target_col='p_pv_W')
cv_results.round(3)

In [ ]:
# Average R² across folds, per model
summary_models = cv_results.groupby(level='model')[['R2', 'CV_RMSE_pct']].mean()
summary_models.sort_values('R2', ascending=False)

## 6 — Final DT-HRES-S selection

Based on these metrics + interpretability + speed, select the best model.
For typical PV regression problems, **Random Forest** tends to win the trade-off:
✓ High R² in cross-city tests  
✓ Robust to outliers  
✓ Interpretable via feature importances  
✓ No scaling required  

Decision Trees are the **interpretable backup** for community workshops.

In [ ]:
# Train the winning RandomForest on ALL data and save it
import joblib
from sklearn.ensemble import RandomForestRegressor

work = ml_models.cyclical_encode(df_all_sim)
features = ml_models.DEFAULT_FEATURES + ['hour_sin', 'hour_cos', 'month_sin',
                                          'month_cos', 'doy_sin', 'doy_cos']
X, y = work[features].dropna(), work.loc[work[features].dropna().index, 'p_pv_W']

final_model = RandomForestRegressor(
    n_estimators=300, max_depth=25, min_samples_leaf=5, n_jobs=-1, random_state=42
)
final_model.fit(X, y)
# joblib.dump(final_model, '../results/models/dt_hres_s_rf_v0.pkl')  # uncomment to save

# Top feature importances
importances = pd.Series(final_model.feature_importances_, index=features).sort_values(ascending=False)
ax = importances.head(10).plot.barh(figsize=(10, 5), color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 10 features driving PV output (Random Forest)')
plt.tight_layout()
plt.show()

## 7 — Use the twin to predict for a **new community** (e.g. Ixil)

Once trained on the four reference cities, the digital twin can be applied to **Ixil, Yucatán** (climate: Aw, tropical savanna — same as Campeche). Just provide the weather TMY for Ixil and the twin predicts the PV output in milliseconds without running the full physics simulation.

In [ ]:
# Demo: use Campeche as a proxy for Ixil and 'predict'
df_ixil_proxy = load_city('Campeche')
df_ixil_enc = ml_models.cyclical_encode(df_ixil_proxy)
X_new = df_ixil_enc[features]
y_pred = final_model.predict(X_new)
print(f'Predicted annual PV generation for Ixil (Campeche proxy): {y_pred.sum()/1000:.0f} kWh/yr')
print(f'Inference time: practically instant (RF on 8,737 rows)')

---
## Next steps for the team

| Task | Lead | What to do |
|---|---|---|
| 3.2 — Collect Ixil-specific data | Samuel | Get a real TMY for Ixil, real load curves |
| 3.3 — Tune NN hyperparameters | José Llashag | The default MLP is undertrained, try wider nets |
| 3.4 — Physics-consistency checks | Regina, Miguel | Verify ML predictions respect energy balance |
| 3.1 — Colab UX | Aaron, Arturo | Add ipywidgets sliders for system sizing |
| Documentation | Carlos | Write the technical report from this notebook |

---
*Made with ❤ by the DT-HRES-S team — Tec de Monterrey, EPICS in IEEE 2025-2026.*